# Imagined-speech EEG: 2–3 word classifier → speech
Pick 2–3 words, compare **12 ML algorithms** on **unseen subjects**, then speak the prediction of the model you choose.

Set `TASK`, `WORDS` and `MODEL` in the config cell. Chance is 50% (2 words) / 33% (3 words). Expect roughly 65–80% for the best word pair and ~45–58% for 3 words — modest, and best-pair numbers are optimistic (see notes).

In [ ]:
!pip -q install mne gTTS

In [ ]:
from google.colab import files
files.upload()   # upload Imagined_speech_EEG_edf.zip
!unzip -q -o Imagined_speech_EEG_edf.zip -d /content

In [ ]:
# ===== CONFIG =====
TASK  = 'Image'                    # 'Image' (words), 'Digit', or 'Char'
WORDS = ['Car', 'Scooter']         # 2 or 3 classes. e.g. ['Car','Scooter','Watch'] | Digit: ['4','9'] | Char: ['A','S']
MODEL = 'SVM (RBF)'                # model used for all detailed plots + prediction (see comparison table below)
T_START, T_END = 0, 12            # seconds of each trial to use (try 3–12 to skip the start-of-trial transient, see sanity check)
N_PERM = 100                       # permutations for the permutation test (lower = faster, esp. for tree models)
# ==================

In [ ]:
import mne, glob, re, itertools, numpy as np, warnings
from scipy.signal import butter, filtfilt, welch
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
mne.set_log_level('ERROR'); warnings.filterwarnings('ignore')

DATA = '/content/Imagined_speech_EEG_edf'
CH = ['AF3','F7','F3','FC5','T7','P7','O1','O2','P8','T8','FC6','F4','F8','AF4']
FS, N = 128, 1536

def load(cat):
    X, y, g = [], [], []
    for f in sorted(glob.glob(f'{DATA}/{cat}/*.edf')):
        m = re.match(r'.*name(\d+)_(.+)\.edf', f)
        d = mne.io.read_raw_edf(f, preload=True).get_data(picks=CH) * 1e6
        d = d[:, :N] if d.shape[1] >= N else np.pad(d, ((0,0),(0,N-d.shape[1])), 'edge')
        X.append(d); y.append(m[2].capitalize() if cat == 'Image' else m[2]); g.append(int(m[1]))
    return np.array(X), np.array(y), np.array(g)

def features(X):
    b, a = butter(4, [1, 45], btype='band', fs=FS)
    X = filtfilt(b, a, X - X.mean(-1, keepdims=True), axis=-1)
    f, P = welch(X, fs=FS, nperseg=128, axis=-1)
    bands = [(1,4),(4,8),(8,13),(13,20),(20,30),(30,45)]
    return np.concatenate([np.log(P[..., (f>=lo)&(f<hi)].mean(-1)+1e-8) for lo,hi in bands], -1).reshape(len(X), -1)

def subject_norm(F, g):
    F = F.copy()
    for s in np.unique(g):
        m = g == s; F[m] = (F[m]-F[m].mean(0)) / (F[m].std(0)+1e-8)
    return F


X, y, g = load(TASK)
F = subject_norm(features(X[..., int(T_START*FS):int(T_END*FS)]), g)   # only the chosen time window is used for classification
print('Available classes:', sorted(set(y)))

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, f_classif

def get_models():
    sc = StandardScaler
    return {
        'SVM (RBF)':           make_pipeline(sc(), SVC(C=1, probability=True, random_state=0)),
        'SVM (Linear)':        make_pipeline(sc(), SVC(kernel='linear', C=0.01, probability=True, random_state=0)),
        'Logistic Regression': make_pipeline(sc(), LogisticRegression(C=0.05, max_iter=3000)),
        'LDA (shrinkage)':     make_pipeline(sc(), LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')),
        'Naive Bayes':         make_pipeline(sc(), GaussianNB()),
        'k-NN':                make_pipeline(sc(), KNeighborsClassifier(7, weights='distance')),
        'Random Forest':       RandomForestClassifier(200, random_state=0),
        'Extra Trees':         ExtraTreesClassifier(200, random_state=0),
        'Gradient Boosting':   HistGradientBoostingClassifier(max_iter=80, learning_rate=0.05, max_depth=3, min_samples_leaf=4, random_state=0),
        'MLP (neural net)':    make_pipeline(sc(), MLPClassifier((64,), alpha=1.0, max_iter=800, random_state=0)),
        'Top-20 features + LogReg': make_pipeline(sc(), SelectKBest(f_classif, k=20), LogisticRegression(C=0.1, max_iter=3000)),
        'Ensemble (soft vote)': VotingClassifier([
            ('svm', make_pipeline(sc(), SVC(C=1, probability=True, random_state=0))),
            ('lr',  make_pipeline(sc(), LogisticRegression(C=0.05, max_iter=3000))),
            ('rf',  RandomForestClassifier(200, random_state=0)),
            ('et',  ExtraTreesClassifier(200, random_state=0))], voting='soft'),
    }
MODELS = get_models()
assert MODEL in MODELS, f'MODEL must be one of {list(MODELS)}'
def make_model(name=None): return clone(MODELS[name or MODEL])

In [ ]:
# ---- Evaluate the chosen words on unseen subjects (repeated subject-wise CV) ----
m = np.isin(y, WORDS); Fm, ym, gm = F[m], y[m], g[m]
accs = []
for rep in range(10):
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(Fm, ym, gm):
        accs.append((make_model().fit(Fm[tr], ym[tr]).predict(Fm[te]) == ym[te]).mean())
print(f'{WORDS}  [{MODEL}] accuracy on unseen subjects = {np.mean(accs):.1%} ± {np.std(accs):.1%}  (chance {1/len(WORDS):.0%})')

## Compare all algorithms
Each model is scored with repeated **subject-wise** cross-validation (no subject appears in both train and test). Differences of a few points are within noise — with 23 subjects the fold-to-fold std is ~10 points. Simple, well-regularised models (SVM-RBF, Naive Bayes) usually match or beat the heavy ones on this little data.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, time
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
K = len(WORDS); REPS = 5
Fm_, ym_, gm_ = Fm, ym, gm
rows, fold_scores = [], {}
for name in MODELS:
    t0 = time.time(); a = []
    for rep in range(REPS):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(Fm_, ym_, gm_):
            a.append((make_model(name).fit(Fm_[tr], ym_[tr]).predict(Fm_[te]) == ym_[te]).mean())
    fold_scores[name] = a
    rows.append({'Model': name, 'Accuracy': np.mean(a), 'Std (folds)': np.std(a), 'Time (s)': round(time.time() - t0, 1)})
res = pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
display(res.style.format({'Accuracy': '{:.1%}', 'Std (folds)': '{:.1%}'}).background_gradient(subset=['Accuracy'], cmap='Greens'))
print(f'Best: {res.Model[0]}  |  chance = {1/K:.0%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
d = res.iloc[::-1]
axes[0].barh(d['Model'], d['Accuracy'], xerr=d['Std (folds)'], color='steelblue', capsize=3)
axes[0].axvline(1 / K, color='r', ls='--', label='chance'); axes[0].set_xlim(0, 1); axes[0].legend()
axes[0].set_xlabel('Accuracy on unseen subjects'); axes[0].set_title(f'Model comparison — {WORDS}')
axes[1].boxplot([fold_scores[n] for n in res.Model]); axes[1].set_xticklabels(res.Model, rotation=40, ha='right', fontsize=8)
axes[1].axhline(1 / K, color='r', ls='--'); axes[1].set_ylabel('Fold accuracy'); axes[1].set_title('Spread across folds')
plt.tight_layout(); plt.show()
print("To use the winner everywhere below, set MODEL = '%s' in the config cell and re-run from there." % res.Model[0])

In [ ]:
# Are the model rankings stable? Accuracy of the fast models on several word pairs (Image words shown; edit `pairs`)
fast = ['SVM (RBF)', 'Logistic Regression', 'LDA (shrinkage)', 'Naive Bayes', 'k-NN', 'Top-20 features + LogReg']
labs_all = sorted(set(y))
pairs = [('Car', 'Scooter'), ('Scooter', 'Watch'), ('Apple', 'Dog'), ('Gold', 'Rose'), ('Tiger', 'Wallet')] if TASK == 'Image' else \
        list(itertools.combinations(labs_all, 2))[:6]
H = np.zeros((len(fast), len(pairs)))
for j, p in enumerate(pairs):
    mm = np.isin(y, p)
    for i, n in enumerate(fast):
        a = [(make_model(n).fit(F[mm][tr], y[mm][tr]).predict(F[mm][te]) == y[mm][te]).mean()
             for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(F[mm], y[mm], g[mm])]
        H[i, j] = np.mean(a)
fig, ax = plt.subplots(figsize=(1.4 * len(pairs) + 3, 4))
im = ax.imshow(H, cmap='viridis', vmin=.4, vmax=.85)
ax.set_xticks(range(len(pairs))); ax.set_xticklabels([' vs '.join(p) for p in pairs], rotation=30, ha='right')
ax.set_yticks(range(len(fast))); ax.set_yticklabels(fast)
for i in range(H.shape[0]):
    for j in range(H.shape[1]): ax.text(j, i, f'{H[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.colorbar(im, label='accuracy (chance 0.50)'); ax.set_title('Model × word-pair accuracy'); plt.tight_layout(); plt.show()

## Sanity check: is the classifier just detecting a start-of-trial artifact?
In this data some words (notably *Scooter*) show a large burst in the first 1–3 s of the trial (movement/blink/onset transient). A classifier can use that instead of imagined speech. Below, the same model is scored using different parts of the 12 s trial. If accuracy **collapses** once the start is removed, the signal was mostly artifact; if it **stays above chance**, there is something more.

In [ ]:
windows = [(0, 12), (0, 3), (3, 12), (5, 12), (6, 12)]
accs = []
for a_, b_ in windows:
    Fw = subject_norm(features(X[..., int(a_*FS):int(b_*FS)]), g)[m]
    s_ = []
    for rep in range(5):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(Fw, ym, gm):
            s_.append((make_model().fit(Fw[tr], ym[tr]).predict(Fw[te]) == ym[te]).mean())
    accs.append((np.mean(s_), np.std(s_)))
lab = [f'{a_}-{b_}s' for a_, b_ in windows]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(lab, [a for a, _ in accs], yerr=[s for _, s in accs], color='steelblue', capsize=3)
ax.axhline(1 / len(WORDS), color='r', ls='--', label='chance'); ax.set_ylim(0, 1); ax.legend()
ax.set_ylabel('Accuracy (unseen subjects)'); ax.set_title(f'{WORDS} — accuracy by time window [{MODEL}]')
for k, (a, _) in enumerate(accs): ax.text(k, a + .02, f'{a:.0%}', ha='center')
plt.tight_layout(); plt.show()

# RMS amplitude first 2 s vs rest, per word: a big ratio = start-of-trial transient
Xf_all = filtfilt(*butter(4, [1, 45], btype='band', fs=FS), X[m] - X[m].mean(-1, keepdims=True), axis=-1)
for c in WORDS:
    sel = ym == c
    print(f'{c:10s} RMS first 2 s = {np.sqrt((Xf_all[sel][..., :256]**2).mean()):5.1f} µV | rest = {np.sqrt((Xf_all[sel][..., 256:]**2).mean()):5.1f} µV')

## Plots 1: what the data looks like

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneGroupOut
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

classes = list(WORDS); K = len(classes)
colors = plt.cm.tab10(np.arange(K))
BANDS = ['delta 1-4', 'theta 4-8', 'alpha 8-13', 'beta-lo 13-20', 'beta-hi 20-30', 'gamma 30-45']

b_, a_ = butter(4, [1, 45], btype='band', fs=FS)
Xm = filtfilt(b_, a_, X[m] - X[m].mean(-1, keepdims=True), axis=-1)   # filtered trials for chosen words
Fz = Fm.reshape(len(Fm), 14, 6)                                        # z-scored band power (trial, channel, band)
t = np.arange(N) / FS

# (a) example filtered trial per class, 4 frontal/temporal channels
show = ['AF3', 'F7', 'T7', 'O1']
fig, axes = plt.subplots(K, 1, figsize=(11, 2.6 * K), sharex=True)
axes = np.atleast_1d(axes)
for ax, c, col in zip(axes, classes, colors):
    i = np.where(ym == c)[0][0]
    for j, ch in enumerate(show):
        ax.plot(t, Xm[i, CH.index(ch)] + j * 80, color=col, lw=.7)
    ax.set_yticks([j * 80 for j in range(len(show))]); ax.set_yticklabels(show)
    ax.set_title(f'Example trial: "{c}" (subject {gm[i]})', loc='left', fontsize=10)
axes[-1].set_xlabel('Time (s)'); plt.tight_layout(); plt.show()

# (b) mean power spectrum per class (average over channels, ±SEM across trials)
f, P = welch(Xm, fs=FS, nperseg=256, axis=-1)
P = P.mean(1)                                                          # trial x freq
fig, ax = plt.subplots(figsize=(7, 4))
for c, col in zip(classes, colors):
    Pc = 10 * np.log10(P[ym == c]); mu = Pc.mean(0); se = Pc.std(0) / np.sqrt(len(Pc))
    ax.plot(f, mu, color=col, label=c); ax.fill_between(f, mu - se, mu + se, color=col, alpha=.2)
ax.set_xlim(1, 45); ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Power (dB, µV²/Hz)')
ax.set_title('Average spectrum per class'); ax.legend(); plt.tight_layout(); plt.show()

# (c) band power per class (z-scored within subject, averaged over channels)
fig, ax = plt.subplots(figsize=(8, 4)); w = .8 / K
for k, (c, col) in enumerate(zip(classes, colors)):
    v = Fz[ym == c].mean(1)                                            # trial x band
    ax.bar(np.arange(6) + k * w, v.mean(0), w, yerr=v.std(0) / np.sqrt(len(v)), color=col, label=c, capsize=2)
ax.set_xticks(np.arange(6) + w * (K - 1) / 2); ax.set_xticklabels(BANDS, rotation=20)
ax.axhline(0, color='k', lw=.6); ax.set_ylabel('Band power (z-score)')
ax.set_title('Band power per class'); ax.legend(); plt.tight_layout(); plt.show()

## Plots 2: where and in which band do classes differ?

In [ ]:
# Topomaps: class-mean band power (z-scored) for every class x band
info = mne.create_info(CH, FS, 'eeg'); info.set_montage('standard_1020')
vmax = np.abs(np.stack([Fz[ym == c].mean(0) for c in classes])).max()
fig, axes = plt.subplots(K, 6, figsize=(13, 2.3 * K + .6))
axes = np.atleast_2d(axes)
for r, c in enumerate(classes):
    mu = Fz[ym == c].mean(0)
    for b in range(6):
        im, _ = mne.viz.plot_topomap(mu[:, b], info, axes=axes[r, b], show=False, vlim=(-vmax, vmax), cmap='RdBu_r')
        if r == 0: axes[r, b].set_title(BANDS[b], fontsize=9)
    axes[r, 0].set_ylabel(c, fontsize=11, rotation=0, labelpad=25)
fig.colorbar(im, ax=axes, shrink=.6, label='z-scored band power'); plt.show()

# PCA of the feature space: is there any class structure at all?
Z = PCA(2).fit_transform(Fm)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for c, col in zip(classes, colors):
    axes[0].scatter(*Z[ym == c].T, color=col, label=c, s=28, alpha=.8)
axes[0].set_title('PCA of features, coloured by word'); axes[0].legend()
sc = axes[1].scatter(*Z.T, c=gm, cmap='nipy_spectral', s=28)
axes[1].set_title('Same points, coloured by subject'); plt.colorbar(sc, ax=axes[1], label='subject id')
for a in axes: a.set_xlabel('PC1'); a.set_ylabel('PC2')
plt.tight_layout(); plt.show()

## Plots 3: how well does the classifier do (unseen subjects)?

In [ ]:
# Collect out-of-fold predictions over repeated subject-wise CV
yt, pr, fold_acc = [], [], []
for rep in range(10):
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=rep).split(Fm, ym, gm):
        mdl = make_model().fit(Fm[tr], ym[tr]); p = mdl.predict_proba(Fm[te])
        yt += list(ym[te]); pr.append(p); fold_acc.append((mdl.classes_[p.argmax(1)] == ym[te]).mean())
yt = np.array(yt); pr = np.vstack(pr); yp = np.array(sorted(classes))[pr.argmax(1)]
cls_sorted = sorted(classes)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# (a) confusion matrix
cm = confusion_matrix(yt, yp, labels=cls_sorted, normalize='true')
ConfusionMatrixDisplay(cm, display_labels=cls_sorted).plot(ax=axes[0], cmap='Blues', values_format='.2f', colorbar=False)
axes[0].set_title(f'Confusion matrix (overall {np.mean(yt == yp):.1%})')
# (b) accuracy across folds vs chance
axes[1].hist(fold_acc, bins=12, color='steelblue', alpha=.85)
axes[1].axvline(1 / K, color='r', ls='--', label=f'chance {1/K:.0%}')
axes[1].axvline(np.mean(fold_acc), color='k', label=f'mean {np.mean(fold_acc):.1%}')
axes[1].set_xlabel('Fold accuracy'); axes[1].set_ylabel('# folds'); axes[1].set_title('Accuracy across CV folds'); axes[1].legend()
# (c) ROC (one-vs-rest)
for k, c in enumerate(cls_sorted):
    fpr, tpr, _ = roc_curve(yt == c, pr[:, k]); axes[2].plot(fpr, tpr, label=f'{c} (AUC {auc(fpr, tpr):.2f})')
axes[2].plot([0, 1], [0, 1], 'k--', lw=.8); axes[2].set_xlabel('False positive rate'); axes[2].set_ylabel('True positive rate')
axes[2].set_title('ROC (one-vs-rest)'); axes[2].legend(loc='lower right')
plt.tight_layout(); plt.show()

# Per-subject accuracy (leave-one-subject-out)
sub_acc = {}
for tr, te in LeaveOneGroupOut().split(Fm, ym, gm):
    sub_acc[gm[te][0]] = (make_model().fit(Fm[tr], ym[tr]).predict(Fm[te]) == ym[te]).mean()
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar([str(s) for s in sub_acc], list(sub_acc.values()), color='steelblue')
ax.axhline(1 / K, color='r', ls='--', label='chance'); ax.axhline(np.mean(list(sub_acc.values())), color='k', label='mean')
ax.set_xlabel('Held-out subject'); ax.set_ylabel('Accuracy'); ax.set_title('Leave-one-subject-out accuracy (few trials per subject → very noisy)')
ax.legend(); plt.tight_layout(); plt.show()

## Plots 4: is the result real? (permutation test + learning curve)
The permutation test shuffles labels within each subject and repeats the whole evaluation. If the real accuracy sits far to the right of the shuffled distribution, the signal is unlikely to be luck.

In [ ]:
rng = np.random.RandomState(0)
def cv_acc(F_, y_, g_, seed=0):
    a = []
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(F_, y_, g_):
        a.append((make_model().fit(F_[tr], y_[tr]).predict(F_[te]) == y_[te]).mean())
    return np.mean(a)

null = []
for _ in range(N_PERM):
    yperm = ym.copy()
    for s in np.unique(gm):
        idx = np.where(gm == s)[0]; yperm[idx] = rng.permutation(ym[idx])
    null.append(cv_acc(Fm, yperm, gm, seed=int(rng.randint(1000))))
obs = np.mean(fold_acc); pval = (np.sum(np.array(null) >= obs) + 1) / (len(null) + 1)

# Learning curve: accuracy vs number of training subjects (test on 5 unseen subjects)
subs = np.unique(gm); sizes = [4, 8, 12, 16, 18]; lc = {n: [] for n in sizes}
for _ in range(30):
    p = rng.permutation(subs); te_s = p[:5]; te_m = np.isin(gm, te_s)
    for n in sizes:
        tr_m = np.isin(gm, p[5:5 + n])
        lc[n].append((make_model().fit(Fm[tr_m], ym[tr_m]).predict(Fm[te_m]) == ym[te_m]).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(null, bins=20, color='grey', alpha=.8, label='shuffled labels')
axes[0].axvline(obs, color='r', lw=2, label=f'real = {obs:.1%} (p = {pval:.3f})')
axes[0].set_xlabel('Accuracy'); axes[0].set_ylabel('count'); axes[0].set_title('Permutation test'); axes[0].legend()
mu = np.array([np.mean(lc[n]) for n in sizes]); sd = np.array([np.std(lc[n]) for n in sizes])
axes[1].plot(sizes, mu, 'o-', color='steelblue'); axes[1].fill_between(sizes, mu - sd, mu + sd, alpha=.2, color='steelblue')
axes[1].axhline(1 / K, color='r', ls='--', label='chance')
axes[1].set_xlabel('# training subjects'); axes[1].set_ylabel('Accuracy on 5 unseen subjects')
axes[1].set_title('Learning curve (still rising = more data would help)'); axes[1].legend()
plt.tight_layout(); plt.show()

### (Optional) find which words work best
Ranking subsets on the same data is optimistic: I measured ~78% this way for a pair that gave ~71% on truly held-out subjects. Use it as a guide, not a final score.

In [ ]:
def rank_subsets(k=2, top=5):
    res = []
    for c in itertools.combinations(sorted(set(y)), k):
        mm = np.isin(y, c); a = []
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(F[mm], y[mm], g[mm]):
            a.append((make_model().fit(F[mm][tr], y[mm][tr]).predict(F[mm][te]) == y[mm][te]).mean())
        res.append((np.mean(a), c))
    for a, c in sorted(res, reverse=True)[:top]: print(f'{a:.1%}', c)
rank_subsets(2)

In [ ]:
# Heatmap of pairwise accuracy for every word pair (optimistic if you pick the max — see note above)
labs = sorted(set(y)); M = np.full((len(labs), len(labs)), np.nan)
for i, j in itertools.combinations(range(len(labs)), 2):
    mm = np.isin(y, [labs[i], labs[j]]); M[i, j] = M[j, i] = cv_acc(F[mm], y[mm], g[mm])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(M, cmap='viridis', vmin=.4, vmax=.85)
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=45, ha='right'); ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs)
for i in range(len(labs)):
    for j in range(len(labs)):
        if i != j: ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.colorbar(im, label='accuracy (chance 0.50)'); ax.set_title('Pairwise 2-word accuracy'); plt.tight_layout(); plt.show()

## Predict on an unseen subject and speak it

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display

test_subj = np.unique(gm)[0]                           # hold out this subject
tr = gm != test_subj; te_idx = np.where(~tr)[0]
clf = make_model().fit(Fm[tr], ym[tr])
for i in te_idx:
    p = clf.predict(Fm[i:i+1])[0]
    print(f'True: {ym[i]:8s} Predicted: {p:8s}', 'OK' if p == ym[i] else 'x')

i = te_idx[0]; pred = clf.predict(Fm[i:i+1])[0]
gTTS(text=f'The imagined word is {pred}', lang='en').save('pred.mp3')
display(Audio('pred.mp3', autoplay=True))

In [ ]:
# Class probabilities the model assigned to the spoken trial
probs = clf.predict_proba(Fm[te_idx[0]:te_idx[0] + 1])[0]
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(clf.classes_, probs, color=[('seagreen' if c == pred else 'lightgrey') for c in clf.classes_])
ax.axhline(1 / K, color='r', ls='--', lw=.8); ax.set_ylim(0, 1); ax.set_ylabel('Probability')
ax.set_title(f'Prediction: {pred}  (true: {ym[te_idx[0]]})'); plt.tight_layout(); plt.show()